# The Transformation Logic

In [0]:
# 1. نكتب اللوجيك كله كـ SQL String مع فلترة التكرار (ROW_NUMBER)
gold_customer_query = """
    WITH RankedCustomers AS (
        SELECT 
            xxhash64(CAST(ci.customer_id AS STRING)) AS customer_key,
            ci.customer_id,
            ci.customer_number,
            ci.first_name,
            ci.last_name,
            la.country,
            ci.marital_status,
            CASE 
                WHEN ci.gender <> 'n/a' AND ci.gender IS NOT NULL THEN ci.gender
                ELSE COALESCE(ca.gender, 'n/a')
            END AS gender,
            ca.birth_date AS birthdate,
            ci.created_date AS create_date,
            
            -- إضافة الترقيم لاكتشاف أحدث نسخة للعميل وحذف التكرار
            ROW_NUMBER() OVER(PARTITION BY ci.customer_id ORDER BY ci.created_date DESC) as row_num
            
        FROM workspace.silver.crm_customers ci
        LEFT JOIN workspace.silver.erp_customers ca
            ON ci.customer_number = ca.customer_number
        LEFT JOIN workspace.silver.erp_customer_location la
            ON ci.customer_number = la.customer_number
    )
    
    -- استدعاء الداتا النظيفة فقط (النسخة رقم 1 الأحدث لكل عميل)
    SELECT 
        customer_key,
        customer_id,
        customer_number,
        first_name,
        last_name,
        country,
        marital_status,
        gender,
        birthdate,
        create_date
    FROM RankedCustomers
    WHERE row_num = 1
"""

# 2. ننفذ الاستعلام ونخزنه في DataFrame
df = spark.sql(gold_customer_query)

# 3. عرض النتيجة للتأكد
display(df)

# 4. كود الحفظ في طبقة الجولد (تأكد إنك بتعمل Overwrite عشان تمسح الداتا القديمة اللي كان فيها تكرار)
# df.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.gold_dim_customers")

# Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

## Sanity checks of Gold table

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customers LIMIT 10